In [102]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [103]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Setup and authentication complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Setup and authentication complete.


In [104]:
import uuid
from google.genai import types
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.tool_context import ToolContext
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

from google.adk.apps.app import App, ResumabilityConfig
from google.adk.tools.function_tool import FunctionTool

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [105]:
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

In [106]:
from google.genai import types

def check_for_approval(events):
    """Detect if agent paused for human approval"""
    for event in events:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if (part.function_call and 
                    part.function_call.name == "adk_request_confirmation"):
                    return {
                        "approval_id": part.function_call.id,
                        "invocation_id": event.invocation_id,
                    }
    return None

def print_agent_response(events):
    """Print only text responses from agent"""
    for event in events:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"Compliance AI > {part.text}")

def create_approval_response(approval_info, approved):
    """Format human yes/no decision for ADK"""
    confirmation_response = types.FunctionResponse(
        id=approval_info["approval_id"],
        name="adk_request_confirmation",
        response={"confirmed": approved},
    )
    return types.Content(
        role="user", 
        parts=[types.Part(function_response=confirmation_response)]
    )

print("✅ All helper functions loaded!")

✅ All helper functions loaded!


In [107]:
# ==============================
# SECTION: Risky Trade Approval Tool (Long-Running)
# ==============================

RISK_THRESHOLD_USD = 5000
HIGH_RISK_KEYWORDS = ["PEPE", "SHIB", "DOGE", "WIF", "BONK", "Futures", "Leverage", "100x"]

def approve_crypto_trade(
    asset: str,
    amount_usd: float,
    action: str,  # "buy" or "sell"
    leverage: int = 1,
    tool_context: ToolContext = None
) -> dict:
    """
    Approves or requests approval for a crypto/stock trade.
    High-risk or large trades require human compliance officer approval.
    """
    risk_score = 0
    reasons = []

    # Size risk
    if amount_usd > RISK_THRESHOLD_USD:
        risk_score += 60
        reasons.append(f"Large amount: ${amount_usd:,.2f}")

    # Meme coin / high volatility risk
    if any(keyword.upper() in asset.upper() for keyword in HIGH_RISK_KEYWORDS):
        risk_score += 80
        reasons.append(f"High-risk asset: {asset}")

    # Leverage risk
    if leverage > 5:
        risk_score += 70
        reasons.append(f"High leverage: {leverage}x")

    risk_level = "LOW" if risk_score < 50 else "MEDIUM" if risk_score < 100 else "HIGH"

    # Auto-approve low-risk trades
    if risk_score < 100:
        return {
            "status": "auto_approved",
            "trade_id": f"TRD-{uuid.uuid4().hex[:8].upper()}",
            "asset": asset,
            "amount_usd": amount_usd,
            "action": action,
            "leverage": leverage,
            "risk_score": risk_score,
            "risk_level": risk_level,
            "message": f"✅ Auto-approved low-risk trade",
            "audit_log": f"Auto-approved {action} {asset} ${amount_usd:,.2f} | Risk: {risk_level}"
        }

    # HIGH RISK → Force human compliance officer approval
    if not tool_context.tool_confirmation:
        tool_context.request_confirmation(
            hint=f"🚨 HIGH-RISK TRADE ALERT 🚨\n"
                 f"Asset: {asset}\n"
                 f"Amount: ${amount_usd:,.2f}\n"
                 f"Action: {action.upper()} | Leverage: {leverage}x\n"
                 f"Risk Score: {risk_score}/100 ({risk_level})\n"
                 f"Reasons: {', '.join(reasons)}\n\n"
                 f"Approve this trade as Compliance Officer?",
            payload={
                "asset": asset,
                "amount_usd": amount_usd,
                "action": action,
                "leverage": leverage,
                "risk_score": risk_score,
                "reasons": reasons
            }
        )
        return {
            "status": "pending_compliance_review",
            "message": f"Trade blocked - awaiting compliance officer approval (Risk: {risk_level})"
        }

    # Resumed after human decision
    if tool_context.tool_confirmation.confirmed:
        return {
            "status": "approved_by_compliance",
            "trade_id": f"TRD-{uuid.uuid4().hex[:8].upper()}",
            "approved_by": "Human Compliance Officer",
            "audit_log": f"MANUALLY APPROVED | {action.upper()} {asset} ${amount_usd:,.2f} | Risk: {risk_level} | Reasons: {', '.join(reasons)}"
        }
    else:
        return {
            "status": "rejected_by_compliance",
            "message": "Trade rejected by compliance officer - funds safe 🔒"
        }

print("✅ High-Risk Trading Compliance Tool Ready!")

✅ High-Risk Trading Compliance Tool Ready!


In [108]:
# ==============================
# AGENT with Killer Persona & Instructions
# ==============================

compliance_agent = LlmAgent(
    name="compliance_officer",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""
You are Apex Compliance AI – an institutional-grade AI Financial Compliance Officer for a crypto hedge fund.

Your core mission: Never allow unauthorized high-risk trades.

Rules:
1. Always use the approve_crypto_trade tool for any buy/sell request
2. If status is pending_compliance_review → clearly tell the user: 
   "This trade has been paused for mandatory compliance review due to high risk."
3. After final result:
   - If approved → show trade ID + audit log
   - If rejected → celebrate that funds are protected
   - Always end with risk summary
4. Use professional, calm, institutional tone — like a bank compliance officer
5. Never apologize for blocking risky trades — it's your job to protect capital.
""",
    tools=[FunctionTool(func=approve_crypto_trade)],
)

# Resumable App (same as before — necessary for pause/resume)
compliance_app = App(
    name="apex_compliance_ai",
    root_agent=compliance_agent,
    resumability_config=ResumabilityConfig(is_resumable=True),
)

session_service = InMemorySessionService()
compliance_runner = Runner(app=compliance_app, session_service=session_service)

print("✅ Apex Compliance AI Agent Ready for Deployment!")

✅ Apex Compliance AI Agent Ready for Deployment!


/tmp/ipykernel_48/506578278.py:31: UserWarning: [EXPERIMENTAL] ResumabilityConfig: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  resumability_config=ResumabilityConfig(is_resumable=True),


In [109]:
# ==============================
# Same Workflow Function (rename only)
# ==============================

async def run_compliance_workflow(query: str, auto_approve: bool = True):
    print(f"\n{'='*70}")
    print(f"Trader > {query}\n")

    session_id = f"compliance_{uuid.uuid4().hex[:8]}"
    await session_service.create_session(app_name="apex_compliance_ai", user_id="trader", session_id=session_id)

    query_content = types.Content(role="user", parts=[types.Part(text=query)])
    events = []

    async for event in compliance_runner.run_async(
        user_id="trader", session_id=session_id, new_message=query_content
    ):
        events.append(event)

    approval_info = check_for_approval(events)

    if approval_info:
        decision = "APPROVED ✅" if auto_approve else "REJECTED ❌"
        print(f"⏸️  Compliance Review Required...")
        print(f"👨‍⚖️ Compliance Officer Decision: {decision}\n")

        async for event in compliance_runner.run_async(
            user_id="trader",
            session_id=session_id,
            new_message=create_approval_response(approval_info, auto_approve),
            invocation_id=approval_info["invocation_id"],
        ):
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        print(f"Compliance AI > {part.text}")
    else:
        print_agent_response(events)

    print(f"{'='*70}\n")

In [110]:
# Demo 1: Small safe trade → instantly auto-approved
await run_compliance_workflow("Buy $3000 worth of Bitcoin")

# Demo 2: Dangerous meme coin + leverage → forces human approval (we simulate APPROVE)
await run_compliance_workflow("Buy $15,000 of PEPE with 20x leverage", auto_approve=True)

# Demo 3: High-risk sell → compliance officer rejects (we simulate REJECT)
await run_compliance_workflow("Sell my entire DOGE position for $8,000", auto_approve=False)


Trader > Buy $3000 worth of Bitcoin



Compliance AI > The trade has been auto-approved. Trade ID: TRD-9C705C2D

Audit Log: Auto-approved buy bitcoin $3,000.00 | Risk: LOW

Risk Summary: The requested trade was classified as low-risk and was processed automatically. No further review is required at this time.


Trader > Buy $15,000 of PEPE with 20x leverage



⏸️  Compliance Review Required...
👨‍⚖️ Compliance Officer Decision: APPROVED ✅

Compliance AI > This trade has been approved by a Human Compliance Officer. Trade ID: TRD-B763602C.

Audit Log: MANUALLY APPROVED | BUY PEPE $15,000.00 | Risk: HIGH | Reasons: Large amount: $15,000.00, High-risk asset: PEPE, High leverage: 20x

Risk Summary: The trade involves a significant amount ($15,000.00), a high-risk asset (PEPE), and substantial leverage (20x), necessitating manual review and approval by a Human Compliance Officer.


Trader > Sell my entire DOGE position for $8,000



⏸️  Compliance Review Required...
👨‍⚖️ Compliance Officer Decision: REJECTED ❌

Compliance AI > The trade has been rejected by compliance.

**Risk Summary:** The requested sale of DOGE was flagged as high-risk due to its potential volatility and the asset's susceptibility to market manipulation. To protect fund assets, this trade was not approved.



In [111]:
await run_compliance_workflow("Emergency: Liquidate all SHIB holdings worth $49,500 right now!", auto_approve=False)


Trader > Emergency: Liquidate all SHIB holdings worth $49,500 right now!



⏸️  Compliance Review Required...
👨‍⚖️ Compliance Officer Decision: REJECTED ❌

Compliance AI > This trade has been rejected by compliance. We successfully protected your funds by preventing this high-risk liquidation.

Risk Summary: SHIB is a volatile meme coin. A large liquidation of $49,500 carries significant market risk, including potential price impact and slippage, which could lead to substantial losses.



In [112]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import uuid
from datetime import datetime

# Risk calculation logic
RISK_THRESHOLD_USD = 5000
HIGH_RISK_KEYWORDS = ["PEPE", "SHIB", "DOGE", "WIF", "BONK", "Futures", "Leverage", "100x"]

def parse_trade_query(query):
    """Parse trade query to extract details"""
    query_lower = query.lower()
    
    # Extract action
    action = "buy" if "buy" in query_lower else "sell" if "sell" in query_lower else "buy"
    
    # Extract amount
    import re
    amount_match = re.search(r'\$?([\d,]+(?:\.\d+)?)', query)
    amount = float(amount_match.group(1).replace(',', '')) if amount_match else 1000
    
    # Extract asset
    words = query.split()
    asset = "BTC"
    for word in words:
        clean_word = re.sub(r'[^A-Z]', '', word.upper())
        if clean_word and 2 <= len(clean_word) <= 5:
            if clean_word not in ['BUY', 'SELL', 'WORTH', 'WITH', 'THE']:
                asset = clean_word
                break
    
    # Extract leverage
    leverage_match = re.search(r'(\d+)x', query, re.IGNORECASE)
    leverage = int(leverage_match.group(1)) if leverage_match else 1
    
    return {
        'asset': asset,
        'amount': amount,
        'action': action,
        'leverage': leverage
    }

def calculate_risk(asset, amount, leverage):
    """Calculate risk score and level"""
    risk_score = 0
    reasons = []
    
    if amount > RISK_THRESHOLD_USD:
        risk_score += 60
        reasons.append(f"Large amount: ${amount:,.2f}")
    
    if any(keyword.upper() in asset.upper() for keyword in HIGH_RISK_KEYWORDS):
        risk_score += 80
        reasons.append(f"High-risk asset: {asset}")
    
    if leverage > 5:
        risk_score += 70
        reasons.append(f"High leverage: {leverage}x")
    
    risk_level = "LOW" if risk_score < 50 else "MEDIUM" if risk_score < 100 else "HIGH"
    
    return {
        'score': risk_score,
        'level': risk_level,
        'reasons': reasons
    }

# UI Components
style = """
<style>
    .apex-header {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        padding: 30px;
        border-radius: 15px;
        text-align: center;
        margin-bottom: 20px;
        box-shadow: 0 10px 40px rgba(102, 126, 234, 0.3);
    }
    .apex-title {
        font-size: 36px;
        font-weight: bold;
        margin: 0;
        text-shadow: 2px 2px 4px rgba(0,0,0,0.2);
    }
    .apex-subtitle {
        font-size: 16px;
        margin-top: 10px;
        opacity: 0.9;
    }
    .trade-box {
        background: #f8f9fa;
        border-radius: 10px;
        padding: 20px;
        margin: 15px 0;
        border-left: 5px solid #667eea;
    }
    .risk-low {
        background: #d4edda;
        border-left: 5px solid #28a745;
    }
    .risk-medium {
        background: #fff3cd;
        border-left: 5px solid #ffc107;
    }
    .risk-high {
        background: #f8d7da;
        border-left: 5px solid #dc3545;
    }
    .alert-box {
        background: #ffe6e6;
        border: 2px solid #ff4444;
        border-radius: 10px;
        padding: 20px;
        margin: 15px 0;
        animation: pulse 2s infinite;
    }
    @keyframes pulse {
        0%, 100% { box-shadow: 0 0 0 0 rgba(255, 68, 68, 0.4); }
        50% { box-shadow: 0 0 0 10px rgba(255, 68, 68, 0); }
    }
    .success-box {
        background: #d4edda;
        border: 2px solid #28a745;
        border-radius: 10px;
        padding: 20px;
        margin: 15px 0;
    }
    .trade-id {
        font-family: monospace;
        background: #e9ecef;
        padding: 5px 10px;
        border-radius: 5px;
        display: inline-block;
    }
</style>
"""

# Display header
display(HTML(style))
display(HTML("""
<div class="apex-header">
    <div class="apex-title">🛡️ Apex Compliance AI</div>
    <div class="apex-subtitle">Institutional-Grade Trading Compliance System</div>
</div>
"""))

# Create widgets
query_input = widgets.Textarea(
    placeholder='Example: Buy $15,000 of PEPE with 20x leverage',
    description='Trade:',
    layout=widgets.Layout(width='100%', height='80px'),
    style={'description_width': '60px'}
)

submit_btn = widgets.Button(
    description='Submit Trade',
    button_style='primary',
    icon='check',
    layout=widgets.Layout(width='200px', height='40px')
)

approve_btn = widgets.Button(
    description='✅ APPROVE',
    button_style='success',
    layout=widgets.Layout(width='150px', height='45px')
)

reject_btn = widgets.Button(
    description='❌ REJECT',
    button_style='danger',
    layout=widgets.Layout(width='150px', height='45px')
)

output_area = widgets.Output()
approval_area = widgets.Output()

# Trade history
trade_history = []

def display_trade_history():
    """Display trade history"""
    with output_area:
        if trade_history:
            print("\n" + "="*70)
            print("📊 TRADE HISTORY")
            print("="*70)
            for i, trade in enumerate(reversed(trade_history[-5:]), 1):
                risk_class = f"risk-{trade['risk']['level'].lower()}"
                status_emoji = "✅" if trade['status'] in ['auto_approved', 'approved_by_compliance'] else "❌"
                print(f"\n{i}. {status_emoji} {trade['action'].upper()} {trade['asset']} - ${trade['amount']:,.2f}")
                print(f"   Risk: {trade['risk']['level']} | Time: {trade['timestamp']}")
                if trade.get('trade_id'):
                    print(f"   ID: {trade['trade_id']}")

def on_submit_clicked(b):
    """Handle submit button click"""
    global pending_trade
    
    with output_area:
        clear_output(wait=True)
        
        query = query_input.value.strip()
        if not query:
            display(HTML('<div class="trade-box">⚠️ Please enter a trade query</div>'))
            return
        
        print("🔄 Processing trade request...\n")
        
        # Parse query
        trade = parse_trade_query(query)
        risk = calculate_risk(trade['asset'], trade['amount'], trade['leverage'])
        
        trade['risk'] = risk
        trade['timestamp'] = datetime.now().strftime("%H:%M:%S")
        
        # Display trade details
        display(HTML(f"""
        <div class="trade-box">
            <h3>📋 Trade Details</h3>
            <p><strong>Asset:</strong> {trade['asset']}</p>
            <p><strong>Amount:</strong> ${trade['amount']:,.2f}</p>
            <p><strong>Action:</strong> {trade['action'].upper()}</p>
            <p><strong>Leverage:</strong> {trade['leverage']}x</p>
            <p><strong>Risk Score:</strong> {risk['score']}/100 ({risk['level']})</p>
        </div>
        """))
        
        if risk['score'] < 100:
            # Auto-approve
            trade_id = f"TRD-{uuid.uuid4().hex[:8].upper()}"
            trade['trade_id'] = trade_id
            trade['status'] = 'auto_approved'
            trade_history.append(trade)
            
            display(HTML(f"""
            <div class="success-box">
                <h3>✅ Trade Auto-Approved</h3>
                <p><strong>Trade ID:</strong> <span class="trade-id">{trade_id}</span></p>
                <p><strong>Status:</strong> Low risk trade automatically approved</p>
                <p><strong>Audit Log:</strong> Auto-approved {trade['action']} {trade['asset']} ${trade['amount']:,.2f} | Risk: {risk['level']}</p>
            </div>
            """))
            
            display_trade_history()
        else:
            # High risk - needs approval
            pending_trade = trade
            
            display(HTML(f"""
            <div class="alert-box">
                <h3>🚨 HIGH-RISK TRADE ALERT 🚨</h3>
                <p><strong>Asset:</strong> {trade['asset']}</p>
                <p><strong>Amount:</strong> ${trade['amount']:,.2f}</p>
                <p><strong>Action:</strong> {trade['action'].upper()}</p>
                <p><strong>Leverage:</strong> {trade['leverage']}x</p>
                <p><strong>Risk Score:</strong> {risk['score']}/100 ({risk['level']})</p>
                <p><strong>Risk Factors:</strong></p>
                <ul>
                    {''.join([f'<li>{reason}</li>' for reason in risk['reasons']])}
                </ul>
                <p><strong>⚠️ This trade requires Compliance Officer approval</strong></p>
            </div>
            """))
            
            with approval_area:
                clear_output(wait=True)
                display(widgets.HBox([approve_btn, reject_btn]))

def on_approve_clicked(b):
    """Handle approve button click"""
    global pending_trade
    
    with output_area:
        clear_output(wait=True)
        
        trade_id = f"TRD-{uuid.uuid4().hex[:8].upper()}"
        pending_trade['trade_id'] = trade_id
        pending_trade['status'] = 'approved_by_compliance'
        trade_history.append(pending_trade)
        
        display(HTML(f"""
        <div class="success-box">
            <h3>✅ Trade Approved by Compliance Officer</h3>
            <p><strong>Trade ID:</strong> <span class="trade-id">{trade_id}</span></p>
            <p><strong>Asset:</strong> {pending_trade['asset']}</p>
            <p><strong>Amount:</strong> ${pending_trade['amount']:,.2f}</p>
            <p><strong>Risk Level:</strong> {pending_trade['risk']['level']}</p>
            <p><strong>Audit Log:</strong> MANUALLY APPROVED | {pending_trade['action'].upper()} {pending_trade['asset']} ${pending_trade['amount']:,.2f} | Risk: {pending_trade['risk']['level']}</p>
        </div>
        """))
        
        display_trade_history()
    
    with approval_area:
        clear_output()
    
    pending_trade = None

def on_reject_clicked(b):
    """Handle reject button click"""
    global pending_trade
    
    with output_area:
        clear_output(wait=True)
        
        pending_trade['status'] = 'rejected_by_compliance'
        trade_history.append(pending_trade)
        
        display(HTML(f"""
        <div class="alert-box">
            <h3>❌ Trade Rejected by Compliance Officer</h3>
            <p><strong>Asset:</strong> {pending_trade['asset']}</p>
            <p><strong>Amount:</strong> ${pending_trade['amount']:,.2f}</p>
            <p><strong>Risk Level:</strong> {pending_trade['risk']['level']}</p>
            <p><strong>🔒 Your funds are safe. Trade blocked for risk protection.</strong></p>
        </div>
        """))
        
        display_trade_history()
    
    with approval_area:
        clear_output()
    
    pending_trade = None

# Attach event handlers
submit_btn.on_click(on_submit_clicked)
approve_btn.on_click(on_approve_clicked)
reject_btn.on_click(on_reject_clicked)

# Display UI
display(HTML("""
<div class="trade-box">
    <h4>💡 Example Queries:</h4>
    <ul>
        <li>Buy $3000 worth of Bitcoin</li>
        <li>Sell my entire DOGE position for $8,000</li>
        <li>Buy $15,000 of PEPE with 20x leverage</li>
    </ul>
</div>
"""))

display(query_input)
display(submit_btn)
display(approval_area)
display(output_area)

print("\n✅ Apex Compliance AI is ready! Enter your trade above.")

Textarea(value='', description='Trade:', layout=Layout(height='80px', width='100%'), placeholder='Example: Buy…

Button(button_style='primary', description='Submit Trade', icon='check', layout=Layout(height='40px', width='2…

Output()

Output()


✅ Apex Compliance AI is ready! Enter your trade above.
